In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# RMI Tutorial 02: Simple Route Setting

Now that your project is set up and the API is enabled, let's register your first monitored road segment using the **Roads Selection API**.

### Objectives:
1. **Define** a monitoring segment (Origin/Destination).
2. **Construct** an RMI-compliant JSON payload.
3. **Register** the route via a REST API call.
4. **Verify** the route status.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googlemaps-samples/insights-samples/blob/main/roads_management_insights/tutorials/02_route_setting.ipynb) 
[![Open In Colab Enterprise](https://img.shields.io/badge/Open%20in-Colab%20Enterprise-blue?logo=google-cloud&logoColor=white)](https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Froads_management_insights%2Ftutorials%2F02_route_setting.ipynb) 
[![Open In BigQuery Notebooks](https://img.shields.io/badge/Open%20in-BigQuery%20Notebooks-blue?logo=google-cloud&logoColor=white)](https://console.cloud.google.com/bigquery/import?url=https%3A%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Froads_management_insights%2Ftutorials%2F02_route_setting.ipynb) 
[![View on GitHub](https://img.shields.io/badge/View%20on-GitHub-lightgrey?logo=github&logoColor=white)](https://github.com/googlemaps-samples/insights-samples/blob/main/roads_management_insights/tutorials/02_route_setting.ipynb)


In [ ]:
# Install necessary libraries
!pip install --upgrade requests google-auth

In [ ]:
import requests
import json
import google.auth
import google.auth.transport.requests
from google.colab import auth

auth.authenticate_user()

PROJECT_ID = 'YOUR_PROJECT_ID' # @param {type:"string"}

In [ ]:
!gcloud config set project {PROJECT_ID}
!gcloud config list

## 1. Define the Route
We will define a simple **Origin/Destination (O/D)** dynamic route. 

### Why use O/D Only?
The O/D-only strategy is the simplest way to define a monitoring segment. By providing only the start and end points, you allow RMI to automatically calculate the optimal path between them based on the underlying road network.

**This strategy is best for:**
- **Transit Monitoring**: Defining segments between consecutive bus stops or train stations.
- **Standard Corridors**: Short road segments where the physical path is unambiguous.
- **Rapid Prototyping**: Quickly setting up a large number of segments without needing precise waypoint traces.

If you wish to influence the path selection (e.g., to bias the route toward specific segments), you would instead provide a list of **intermediate waypoints** (up to 25). Note that intermediates act as hints and do not strictly enforce a specific lane or microscopic path.

In [ ]:
ROUTE_ID = 'tutorial-route-001' # No underscores allowed!
DISPLAY_NAME = 'Tutorial Prototype Route'

# Example: Coordinates for a segment in Boston
ORIGIN = {"latitude": 42.3601, "longitude": -71.0589}
DESTINATION = {"latitude": 42.3611, "longitude": -71.0579}

payload = {
    "displayName": DISPLAY_NAME,
    "dynamicRoute": {
        "origin": ORIGIN,
        "destination": DESTINATION,
        "intermediates": []
    },
    "routeAttributes": {
        "use_case": "prototype",
        "tutorial_id": "02"
    }
}

## 2. Register the Route
We use an authenticated POST request to the Roads Selection API. For detailed specifications, refer to the [official guide on selecting roads](https://developers.google.com/maps/documentation/roads-management-insights/roads-selection/create) and the [REST API reference for route creation](https://developers.google.com/maps/documentation/roads-management-insights/reference/rest/v1/selection.v1.projects.selectedRoutes/create).

In [ ]:
# Get Access Token
creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
auth_req = google.auth.transport.requests.Request()
creds.refresh(auth_req)

ROADS_SELECTION_BASE_URL="https://roads.googleapis.com/selection/v1"

url = f"{ROADS_SELECTION_BASE_URL}/projects/{PROJECT_ID}/selectedRoutes?selectedRouteId={ROUTE_ID}"
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {creds.token}",
    "x-goog-user-project": f"{PROJECT_ID}"
}

response = requests.post(url, headers=headers, json=payload)

if response.status_code == 200:
    print("Success! Route registered.")
    print(json.dumps(response.json(), indent=2))
else:
    print(f"Error {response.status_code}: {response.text}")

## 3. Monitor Route Status (Polling)
Once registered, a route initially enters the `STATE_VALIDATING` phase. RMI performs internal geometry checks and verifies road usage. The cell below uses the [Get SelectedRoute REST method](https://developers.google.com/maps/documentation/roads-management-insights/reference/rest/v1/selection.v1.projects.selectedRoutes/get) to poll the API until the state changes to `STATE_RUNNING` or `STATE_INVALID`.

In [ ]:
import time

get_url = f"{ROADS_SELECTION_BASE_URL}/projects/{PROJECT_ID}/selectedRoutes/{ROUTE_ID}"
status = "STATE_VALIDATING"
max_retries = 10
retry_count = 0

print(f"Polling status for {ROUTE_ID}...")

while status == "STATE_VALIDATING" and retry_count < max_retries:
    response = requests.get(get_url, headers=headers)
    if response.status_code == 200:
        data = response.json()
        status = data.get('state', 'UNKNOWN')
        validation_error = data.get('validationError', 'None')
        
        print(f" - Current State: {status} (Validation Error: {validation_error})")
        
        if status != "STATE_VALIDATING":
            break
    else:
        print(f"Error {response.status_code}: {response.text}")
        break
    
    retry_count += 1
    time.sleep(30) # Poll every 30 seconds

if status == "STATE_RUNNING":
    print("\nSuccess! Your route is now active and monitoring traffic.")
elif status == "STATE_INVALID":
    print(f"\nRoute is INVALID. Reason: {validation_error}")
else:
    print("\nPolling timed out or encountered an error.")

## Summary & Next Steps
You have successfully registered your first monitored route. RMI will now begin validating the geometry and traffic volumes.

**Next**: In the next tutorial, we will verify how the data flows into BigQuery once the route is active.

---
**For more advanced RMI query patterns, visit the official [RMI Sample Queries Repository](https://github.com/googlemaps-samples/insights-samples/tree/main/roads_management_insights/rmi_sample_queries).**